# 00_config
Central configuration for the Zip Files copying/ingestion pipeline

### Required Libraries

In [0]:
import os
import uuid
import zipfile
from datetime import datetime
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, TimestampType
)
from pathlib import Path

### Catalog & Schema

In [0]:
catalog = "iics"
schema  = "casemix"

### Table Names 
- `control_table` drives *what* gets copied (source -> target).
- `log_table` captures *what happened* on every run.

In [0]:
control_table_name = "zip_files_and_json_control_test"
log_table_name     = "zip_files_execution_log_test"

control_table = f"{catalog}.{schema}.{control_table_name}"
log_table     = f"{catalog}.{schema}.{log_table_name}"

print(f"control_table = {control_table}")
print(f"log_table     = {log_table}")

### Temp Remove when prod

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {control_table}")
# spark.sql(f"DROP TABLE IF EXISTS {log_table}")

## Run-level settings

In [0]:
# Only files with .zip extensions are considered for copy
valid_extensions = (".zip",)

# Minimum number of underscore-separated tokens a filename must have
min_filename_tokens = 2

# 01_setup_tablesCreates 
The **control table** and the **execution log table**, and seeds the control table the first time it is run.

### Control table

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {control_table} (
    s_no                        INT NOT NULL,
    zip_source                  STRING NOT NULL,
    zip_source_path             STRING NOT NULL,
    zip_tgt                     STRING NOT NULL,
    zip_tgt_path                STRING NOT NULL,
    unzip_tgt_path              STRING NOT NULL,
    file_type                   STRING NOT NULL,
    tgt_catalog                 STRING NOT NULL,
    tgt_schema                  STRING NOT NULL,
    tgt_table                   STRING NOT NULL,
    versioning_partition_filed  STRING NOT NULL,
    is_full_load                INT NOT NULL,
    is_incremental_load         INT NOT NULL,
    is_active                   INT NOT NULL
)
USING DELTA;
""")

### Log table
Added `stage` (`COPY` / `UNZIP`) so a single log table can capture both pipeline stages for a `run_id`.

In [0]:
LOG_SCHEMA = StructType([
    StructField("run_id",      StringType(),    True),
    StructField("stage",       StringType(),    True), 
    StructField("s_no",        IntegerType(),   True),
    StructField("source",      StringType(),    True),
    StructField("file_name",   StringType(),    True),
    StructField("source_path", StringType(),    True),
    StructField("target_path", StringType(),    True),
    StructField("status",      StringType(),    True),
    StructField("message",     StringType(),    True),
    StructField("start_time",  TimestampType(), True),
    StructField("end_time",    TimestampType(), True),
])

In [0]:
log_ddl_columns = ",\n    ".join(
    f"{f.name} {f.dataType.simpleString().upper()}" for f in LOG_SCHEMA.fields
)

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {log_table} (
    {log_ddl_columns}
)
USING DELTA;
""")

### Seed the control table (optional)

In [0]:
spark.sql(f"""
        INSERT INTO {control_table}
        (s_no, zip_source, zip_source_path, zip_tgt, zip_tgt_path, unzip_tgt_path, file_type,tgt_catalog,tgt_schema,tgt_table,versioning_partition_filed,is_full_load,is_incremental_load,is_active)
        VALUES
        (1, 's3', 's3://cittabase-s3-test/Databricks-poc/zip_files/','databricks_volume', '/Volumes/iics/casemix/history/casemix/zip_files/raw/','/Volumes/iics/casemix/history/casemix/zip_files/processed/','json','iics','casemix','json_test','file_name',1,0,1),
        (2, 's3', 's3://cittabase-s3-test/Databricks-poc/zip_files_1/',
         'databricks_volume', '/Volumes/iics/casemix/history/casemix/zip_files/raw/','/Volumes/iics/casemix/history/casemix/zip_files/processed/', 'json','iics','casemix','json_test','file_name',0,0,0)
    """)
print("Seeded control table with 2 rows.")